# HonestForest labeled outlier audit

For class-relative anomaly geometry use an explicitly symmetric relation; reserve `honest_gap` for prediction identity.


All examples use local synthetic data, fixed seeds, and a reduced smoke
configuration by default. `original`, `kerf`, `path`, `random_height`, and the
separately named isolation `collision` relation are symmetric affinities.
`rfgap`, `honest_gap`, and conditional weights are directed query-to-training
weights; downstream visualization, clustering, or attribution must state any
symmetrization. Prediction identity is claimed only for unsymmetrized `rfgap`
or `honest_gap` under their declared bootstrap/honesty contracts.

In [1]:
SMOKE = True
RANDOM_STATE = 0
N_SAMPLES = 80 if SMOKE else 800
N_ESTIMATORS = 20 if SMOKE else 300

In [2]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=N_SAMPLES, n_features=8, n_informative=5,
    n_redundant=1, random_state=RANDOM_STATE,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE,
)

In [3]:
import inspect
from forestgap import Proximity, outlier_scores

try:
    from treeple import HonestForestClassifier
except ImportError:
    HonestForestClassifier = None

if HonestForestClassifier is None:
    result = {"status": "skipped", "reason": "install forestgap[treeple]"}
else:
    signature = inspect.signature(HonestForestClassifier.__init__)
    kwargs = {"n_estimators": N_ESTIMATORS, "random_state": RANDOM_STATE}
    if "n_jobs" in signature.parameters:
        kwargs["n_jobs"] = 1
    forest = HonestForestClassifier(**kwargs).fit(X_train, y_train)
    proximity = Proximity(forest, relation="original").attach(X_train, y_train)
    scores = outlier_scores(proximity, y_train, method="ene", scaling="normalize")
    result = {"status": "ok", "relation": "original", "largest_scores": np.sort(scores)[-5:].round(3).tolist()}
result

{'status': 'ok',
 'relation': 'original',
 'largest_scores': [0.486, 0.518, 0.802, 0.892, 1.0]}